# USGS Spectral Library — Interactive Analysis

Visualise and explore the 35 spectra from `backend/data/spectra.json`  
(ingested from USGS splib07b by `data/ingest_usgs.py`).

**What's here**
- Single-spectrum viewer with diagnostic-feature markers
- Continuum-removed view toggle
- Multi-spectrum overlay for comparison
- Category browser

In [ ]:
import json
from pathlib import Path

import numpy as np
import xarray as xr
import panel as pn
import hvplot.xarray  # noqa: F401  registers .hvplot accessor on DataArray
import holoviews as hv

pn.extension(sizing_mode="stretch_width")

## 1 — Load spectra into an xarray Dataset

In [ ]:
SPECTRA_JSON = Path("../backend/data/spectra.json")

with open(SPECTRA_JSON) as f:
    records = json.load(f)

# ── coordinates ──────────────────────────────────────────────────────────────
wl = np.array(records[0]["wavelengths_nm"], dtype=float)
ids = [r["id"] for r in records]

# ── 2-D reflectance arrays  (spectrum × wavelength) ──────────────────────────
def _arr(key):
    return np.array(
        [[np.nan if v is None else v for v in r[key]] for r in records],
        dtype=float,
    )

ds = xr.Dataset(
    {
        "reflectance": xr.DataArray(
            _arr("reflectance"),
            dims=["spectrum", "wavelength_nm"],
            coords={"spectrum": ids, "wavelength_nm": wl},
            attrs={"long_name": "Absolute reflectance", "units": "fraction"},
        ),
        "reflectance_cr": xr.DataArray(
            _arr("reflectance_cr"),
            dims=["spectrum", "wavelength_nm"],
            coords={"spectrum": ids, "wavelength_nm": wl},
            attrs={"long_name": "Continuum-removed reflectance", "units": "fraction"},
        ),
    },
    attrs={"source": "USGS splib07b"},
)

# ── metadata lookup dicts ─────────────────────────────────────────────────────
meta = {r["id"]: r for r in records}
display_name = {r["id"]: r["display_name"] for r in records}
categories   = sorted({r["category"] for r in records})

print(ds)
print(f"\n{len(ids)} spectra, {len(wl)} wavelength bands ({int(wl[0])}–{int(wl[-1])} nm)")

## 2 — Single-spectrum viewer

Select a spectrum, toggle continuum-removal, and see its diagnostic absorption features overlaid as vertical markers.

In [ ]:
# ── helper: build feature-marker overlay ─────────────────────────────────────
def feature_overlay(spectrum_id: str, use_cr: bool) -> hv.Overlay:
    """Return VLines + Labels for the diagnostic features of one spectrum."""
    features = meta[spectrum_id]["diagnostic_features"]
    if not features:
        return hv.Overlay([])

    var = "reflectance_cr" if use_cr else "reflectance"
    da  = ds[var].sel(spectrum=spectrum_id)

    elements = []
    for feat in features:
        wl_f = feat["wavelength_nm"]
        desc = feat["description"]
        # value of the spectrum at (or near) this wavelength
        y_val = float(da.sel(wavelength_nm=wl_f, method="nearest"))

        vline = hv.VLine(wl_f).opts(
            color="crimson", line_dash="dashed", line_width=1.2, alpha=0.7
        )
        label = hv.Text(
            wl_f, y_val * 1.02 if np.isfinite(y_val) else 0.95,
            f"{wl_f} nm",
            halign="left", valign="bottom",
        ).opts(text_font_size="8pt", text_color="crimson", angle=90)

        tip = hv.Points(
            {"x": [wl_f], "y": [y_val], "desc": [desc]},
            kdims=["x", "y"],
            vdims=["desc"],
        ).opts(
            color="crimson", size=7, marker="triangle",
            tools=["hover"], hover_tooltips=[("wavelength", "@x nm"), ("feature", "@desc")],
        )

        elements += [vline, label, tip]

    return hv.Overlay(elements)


# ── widgets ───────────────────────────────────────────────────────────────────
spectrum_select = pn.widgets.Select(
    name="Spectrum",
    options={display_name[sid]: sid for sid in ids},
    value=ids[0],
    width=320,
)
cr_toggle = pn.widgets.Toggle(
    name="Continuum removed", value=False, button_type="primary", width=180
)


# ── reactive plot ─────────────────────────────────────────────────────────────
@pn.depends(spectrum_select, cr_toggle)
def single_spectrum_plot(spectrum_id, use_cr):
    var   = "reflectance_cr" if use_cr else "reflectance"
    label = "Continuum-removed reflectance" if use_cr else "Reflectance"
    info  = meta[spectrum_id]

    da = ds[var].sel(spectrum=spectrum_id)

    line = da.hvplot.line(
        x="wavelength_nm",
        ylabel=label,
        xlabel="Wavelength (nm)",
        title=f"{info['display_name']}  —  {info['category']} / {info['subcategory']}",
        color="steelblue",
        line_width=1.8,
        height=420,
        tools=["hover", "box_zoom", "reset"],
        hover_tooltips=[
            ("Wavelength", "$x{0} nm"),
            (label, "$y{0.0000}"),
        ],
    )

    plot = (line * feature_overlay(spectrum_id, use_cr)).opts(
        hv.opts.Overlay(legend_position="top_right")
    )
    return plot


@pn.depends(spectrum_select)
def metadata_panel(spectrum_id):
    info = meta[spectrum_id]
    features_md = "\n".join(
        f"- **{f['wavelength_nm']} nm** — {f['description']}"
        for f in info["diagnostic_features"]
    ) or "_No diagnostic features (featureless spectrum)_"

    tags_str = " · ".join(f"`{t}`" for t in info["tags"])
    aliases_str = ", ".join(info["aliases"]) or "—"

    return pn.pane.Markdown(
        f"""### {info['display_name']}
**Source:** {info['source_id']}  
**Tags:** {tags_str}  
**Aliases:** {aliases_str}

#### Diagnostic absorption features
{features_md}

#### Explanation
{info['explanation']}
""",
        width=380,
    )


viewer = pn.Column(
    pn.Row(spectrum_select, cr_toggle),
    pn.Row(
        pn.panel(single_spectrum_plot, width=700),
        pn.panel(metadata_panel, width=400),
    ),
)
viewer.servable()

## 3 — Multi-spectrum comparison

Select several spectra to overlay on the same axes. Useful for distinguishing spectrally similar minerals (e.g. calcite vs dolomite, kaolinite vs muscovite).

In [ ]:
multi_select = pn.widgets.MultiSelect(
    name="Spectra (hold Ctrl/Cmd to select multiple)",
    options={display_name[sid]: sid for sid in ids},
    value=ids[:3],
    size=8,
    width=320,
)
multi_cr_toggle = pn.widgets.Toggle(
    name="Continuum removed", value=False, button_type="primary", width=180
)

# palette with enough colours for all 35 spectra
PALETTE = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
]


@pn.depends(multi_select, multi_cr_toggle)
def multi_spectrum_plot(selected_ids, use_cr):
    if not selected_ids:
        return pn.pane.Markdown("_Select at least one spectrum._")

    var   = "reflectance_cr" if use_cr else "reflectance"
    label = "Continuum-removed reflectance" if use_cr else "Reflectance"

    curves = []
    for i, sid in enumerate(selected_ids):
        da = ds[var].sel(spectrum=sid)
        colour = PALETTE[i % len(PALETTE)]
        curve = da.hvplot.line(
            x="wavelength_nm",
            label=display_name[sid],
            color=colour,
            line_width=1.8,
        )
        curves.append(curve)

    combined = hv.Overlay(curves).opts(
        hv.opts.Curve(tools=["hover"]),
        hv.opts.Overlay(
            title=f"Comparison — {label}",
            xlabel="Wavelength (nm)",
            ylabel=label,
            legend_position="top_right",
            height=480,
            show_grid=True,
        ),
    )
    return combined


comparison = pn.Column(
    pn.Row(multi_select, multi_cr_toggle),
    pn.panel(multi_spectrum_plot),
)
comparison.servable()

## 4 — Category browser

Plot all spectra in a category as a grid (one panel per spectrum, absolute reflectance).

In [ ]:
category_select = pn.widgets.Select(
    name="Category", options=categories, value=categories[0], width=220
)


@pn.depends(category_select)
def category_grid(cat):
    cat_ids = [r["id"] for r in records if r["category"] == cat]
    if not cat_ids:
        return pn.pane.Markdown("_No spectra in this category._")

    plots = []
    for sid in cat_ids:
        da = ds["reflectance"].sel(spectrum=sid)
        p = da.hvplot.line(
            x="wavelength_nm",
            title=display_name[sid],
            xlabel="",
            ylabel="Reflectance",
            color="steelblue",
            line_width=1.5,
            height=220,
            width=480,
            tools=["hover"],
        )
        # add feature markers
        for feat in meta[sid]["diagnostic_features"]:
            p = p * hv.VLine(feat["wavelength_nm"]).opts(
                color="crimson", line_dash="dashed", line_width=1, alpha=0.6
            )
        plots.append(p)

    # arrange into two-column layout
    rows = []
    for i in range(0, len(plots), 2):
        pair = plots[i : i + 2]
        rows.append(pn.Row(*[pn.panel(p) for p in pair]))
    return pn.Column(*rows)


browser = pn.Column(
    category_select,
    pn.panel(category_grid),
)
browser.servable()

---
### Quick reference — all spectrum IDs

In [ ]:
import pandas as pd

summary = pd.DataFrame(
    [
        {
            "id": r["id"],
            "display_name": r["display_name"],
            "category": r["category"],
            "subcategory": r["subcategory"],
            "n_features": len(r["diagnostic_features"]),
            "tags": ", ".join(r["tags"]),
        }
        for r in records
    ]
)
summary